In [8]:
def group_scores_by_model(
    predictions: list[tuple[str, float]]
) -> dict[str, list[float]]:
    results = {}

    for model, score in predictions:
        if model not in results:
            results[model] = []

        results[model].append(score)

    return results
# tests
assert group_scores_by_model([]) == {}

assert group_scores_by_model([
    ("bert", 0.91),
]) == {
    "bert": [0.91],
}

assert group_scores_by_model([
    ("bert", 0.91),
    ("xgboost", 0.82),
    ("bert", 0.87),
    ("random_forest", 0.76),
    ("xgboost", 0.89),
]) == {
    "bert": [0.91, 0.87],
    "xgboost": [0.82, 0.89],
    "random_forest": [0.76],
}

assert group_scores_by_model([
    ("A", 1.0),
    ("A", 2.0),
    ("A", 3.0),
]) == {
    "A": [1.0, 2.0, 3.0],
}

In [17]:
from collections import defaultdict

def group_errors_by_service(
    logs: list[dict]
) -> dict[str, list[str]]:
    results = defaultdict(list)

    for log in logs:
        service = log["service"]
        level = log["level"]
        message = log["message"]
        
        if level != "ERROR":
            continue
        
        results[service].append(message)
    return dict(results)

# Tests
assert group_errors_by_service([]) == {}

assert group_errors_by_service([
    {"service": "api", "level": "INFO", "message": "started"},
]) == {}

assert group_errors_by_service([
    {"service": "api", "level": "ERROR", "message": "timeout"},
]) == {
    "api": ["timeout"],
}

assert group_errors_by_service([
    {"service": "api", "level": "ERROR", "message": "timeout"},
    {"service": "db", "level": "INFO", "message": "connected"},
    {"service": "api", "level": "ERROR", "message": "invalid token"},
    {"service": "db", "level": "ERROR", "message": "connection lost"},
    {"service": "api", "level": "WARNING", "message": "slow response"},
    {"service": "db", "level": "ERROR", "message": "timeout"},
]) == {
    "api": ["timeout", "invalid token"],
    "db": ["connection lost", "timeout"],
}

In [21]:
from collections import defaultdict

def summarize_predictions(
    predictions: list[dict],
) -> dict[str, dict]:
    results = defaultdict(dict)

    for prediction in predictions:
        model = prediction["model"]
        score = prediction["score"]
        correct = prediction["correct"]

        results[model] = {
            "count": results[model].get("count", 0) + 1,
            "score_sum": results[model].get("score_sum", 0) + score,
            "correct_count": results[model].get("correct_count", 0) + int(correct)
        }
    return results

# Tests
assert summarize_predictions([]) == {}

assert summarize_predictions([
    {"model": "bert", "score": 0.5, "correct": False},
]) == {
    "bert": {
        "count": 1,
        "score_sum": 0.5,
        "correct_count": 0,
    }
}

assert summarize_predictions([
    {"model": "A", "score": 1.0, "correct": True},
    {"model": "B", "score": 0.5, "correct": False},
    {"model": "A", "score": 0.8, "correct": False},
    {"model": "B", "score": 0.7, "correct": True},
    {"model": "A", "score": 0.6, "correct": True},
]) == {
    "A": {
        "count": 3,
        "score_sum": 2.4,
        "correct_count": 2,
    },
    "B": {
        "count": 2,
        "score_sum": 1.2,
        "correct_count": 1,
    },
}

In [44]:
from collections import defaultdict

def summarize_incidents(
    incidents: list[dict]
) -> dict[str, dict]:
    results = defaultdict(lambda : 
        {
            "count": 0,
            "total_duration": 0,
            "resolved_count": 0,
            "severity_counts": {
                "LOW": 0,
                "MEDIUM": 0,
                "HIGH": 0
            },
        }
    )
    for incident in incidents:
        
        service = incident["service"]
        severity = incident["severity"]
        duration = incident["duration"]
        resolved = incident["resolved"]
        
        item = results[service]

        item["count"] += 1
        item["total_duration"] += duration
        item["resolved_count"] += int(resolved)
        item["severity_counts"][severity] += 1
    
    return dict(results)

# Tests
incidents = [
    {"service": "api", "severity": "HIGH", "duration": 120, "resolved": True},
    {"service": "database", "severity": "MEDIUM", "duration": 40, "resolved": False},
    {"service": "api", "severity": "LOW", "duration": 15, "resolved": True},
    {"service": "worker", "severity": "HIGH", "duration": 300, "resolved": False},
    {"service": "database", "severity": "HIGH", "duration": 90, "resolved": True},
    {"service": "api", "severity": "HIGH", "duration": 60, "resolved": False},
    {"service": "worker", "severity": "MEDIUM", "duration": 45, "resolved": True},
]

assert summarize_incidents([]) == {}

assert summarize_incidents([
    {
        "service": "api",
        "severity": "LOW",
        "duration": 10,
        "resolved": False,
    }
]) == {
    "api": {
        "count": 1,
        "total_duration": 10,
        "resolved_count": 0,
        "severity_counts": {
            "LOW": 1,
            "MEDIUM": 0,
            "HIGH": 0,
        },
    }
}

assert summarize_incidents(incidents) == {
    "api": {
        "count": 3,
        "total_duration": 195,
        "resolved_count": 2,
        "severity_counts": {
            "LOW": 1,
            "MEDIUM": 0,
            "HIGH": 2,
        },
    },
    "database": {
        "count": 2,
        "total_duration": 130,
        "resolved_count": 1,
        "severity_counts": {
            "LOW": 0,
            "MEDIUM": 1,
            "HIGH": 1,
        },
    },
    "worker": {
        "count": 2,
        "total_duration": 345,
        "resolved_count": 1,
        "severity_counts": {
            "LOW": 0,
            "MEDIUM": 1,
            "HIGH": 1,
        },
    },
}